In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Load the training CSV from the Kaggle dataset
df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "simhadrisadaram/mimic-cxr-dataset",
    "mimic_cxr_aug_train.csv"
)

print("Shape:", df.shape)
print(df.head())
print(df.columns.tolist())

In [ ]:
# Check missing values
print(df.isnull().sum())

# Inspect the original report and augmented report
for col in ["text", "text_augment"]:
    if col in df.columns:
        print(f"\n--- {col} ---")
        print(df[col].iloc[0])

In [ ]:

import pandas as pd
import ast
from collections import Counter



def parse_list(value):
    """Convert stringified lists into Python lists safely."""
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == "":
        return []
    try:
        result = ast.literal_eval(value)
        return result if isinstance(result, list) else []
    except (ValueError, SyntaxError):
        return []

# Parse image path columns
list_cols = ["image", "view", "AP", "PA", "Lateral"]
for col in list_cols:
    df[col] = df[col].apply(parse_list)

# 1. Count image paths in each column
for col in ["image", "AP", "PA", "Lateral"]:
    paths = [
        str(path)
        for items in df[col]
        for path in items
        if isinstance(path, str) and path.strip()
    ]
    print(f"{col}: {len(paths):,} path occurrences")
    print(f"{col} unique paths: {len(set(paths)):,}")

# 2. Count view labels
view_counts = Counter(
    str(view).strip().upper()
    for views in df["view"]
    for view in views
)

print("\nView label counts:")
for view, count in view_counts.most_common():
    print(f"{view}: {count:,}")

# 3. Count records containing frontal and lateral images
df["has_AP"] = df["AP"].apply(len).gt(0)
df["has_PA"] = df["PA"].apply(len).gt(0)
df["has_Lateral"] = df["Lateral"].apply(len).gt(0)

df["has_frontal"] = df["has_AP"] | df["has_PA"]

print("\nRecord-level summary:")
print("Total rows:", len(df))
print("Unique subjects:", df["subject_id"].nunique())
print("Records with AP:", df["has_AP"].sum())
print("Records with PA:", df["has_PA"].sum())
print("Records with Lateral:", df["has_Lateral"].sum())
print("Records with frontal:", df["has_frontal"].sum())
print("Frontal + lateral:", (df["has_frontal"] & df["has_Lateral"]).sum())
print("Frontal only:", (df["has_frontal"] & ~df["has_Lateral"]).sum())
print("Lateral only:", (~df["has_frontal"] & df["has_Lateral"]).sum())
print("Neither in AP/PA/Lateral columns:",
      (~df["has_frontal"] & ~df["has_Lateral"]).sum())

# 4. Count unique image paths across all view-specific columns
all_view_paths = set()
for col in ["AP", "PA", "Lateral"]:
    for paths in df[col]:
        all_view_paths.update(
            p for p in paths if isinstance(p, str) and p.strip()
        )

print("\nUnique paths across AP, PA, Lateral:",
      len(all_view_paths))


In [ ]:
import os
import re
import ast
import pandas as pd
import numpy as np

BASE = "/kaggle/input/datasets/simhadrisadaram/mimic-cxr-dataset"

TRAIN_CSV = os.path.join(BASE, "mimic_cxr_aug_train.csv")
VAL_CSV = os.path.join(BASE, "mimic_cxr_aug_validate.csv")

IMAGE_ROOT = os.path.join(BASE, "official_data_iccv_final")
OUTPUT_DIR = "/kaggle/working/processed_chest_xray"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for path in [TRAIN_CSV, VAL_CSV, IMAGE_ROOT]:
    print(path, "->", os.path.exists(path))

if not os.path.isfile(TRAIN_CSV) or not os.path.isfile(VAL_CSV):
    print("\nCSV path not found. Available CSV files:")
    for root, dirs, files in os.walk("/kaggle/input"):
        for filename in files:
            if filename.endswith(".csv"):
                print(os.path.join(root, filename))

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

display(train_df.head(3))

In [ ]:
def parse_list(value):
    if isinstance(value, list):
        return value

    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []

    if not isinstance(value, str) or not value.strip():
        return []

    try:
        result = ast.literal_eval(value)
        return result if isinstance(result, list) else [result]
    except (ValueError, SyntaxError):
        return [value]


def get_path_list(value):
    return [
        str(p).strip()
        for p in parse_list(value)
        if isinstance(p, str) and p.strip()
    ]


def normalize_image_path(path):
    path = str(path).strip().replace("\\", "/")

    # استخدم الجزء الذي يبدأ بـ files/ لو المسار قديمًا أو مطلقًا
    match = re.search(r"(?:^|/)files/(.+)$", path)

    if match:
        return os.path.normpath(
            os.path.join(IMAGE_ROOT, "files", match.group(1))
        )

    if os.path.isabs(path) and os.path.exists(path):
        return path

    return os.path.normpath(os.path.join(IMAGE_ROOT, path))


def get_study_id(path):
    match = re.search(r"(?:^|/)s(\d+)(?:/|$)", path.replace("\\", "/"))
    return match.group(1) if match else ""


def clean_report(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""

    # التقارير قد تكون محفوظة كقائمة نصوص
    if isinstance(value, str):
        value = value.strip()
        if not value:
            return ""

        parsed = parse_list(value)
        if parsed and all(isinstance(x, str) for x in parsed):
            return " ".join(x.strip() for x in parsed if x.strip())

        return value

    if isinstance(value, list):
        return " ".join(str(x).strip() for x in value if str(x).strip())

    return str(value).strip()


def process_dataframe(data, split_name):
    required = ["subject_id", "AP", "PA", "Lateral", "text"]
    missing = [c for c in required if c not in data.columns]

    if missing:
        raise ValueError(
            f"{split_name}: missing columns {missing}. "
            f"Available columns: {data.columns.tolist()}"
        )

    records = []

    for _, row in data.iterrows():
        ap = [normalize_image_path(p) for p in get_path_list(row["AP"])]
        pa = [normalize_image_path(p) for p in get_path_list(row["PA"])]
        lat = [normalize_image_path(p) for p in get_path_list(row["Lateral"])]

        paths = ap + pa + lat
        study_ids = {get_study_id(p) for p in paths if get_study_id(p)}

        # لا نخمن عند وجود أكثر من Study في الصف
        if len(study_ids) != 1:
            continue

        study_id = next(iter(study_ids))

        def choose(paths):
            return next(
                (p for p in paths if get_study_id(p) == study_id and os.path.isfile(p)),
                ""
            )

        # PA أولًا، ثم AP كبديل
        frontal_path = choose(pa)
        frontal_view = "PA" if frontal_path else ""

        if not frontal_path:
            frontal_path = choose(ap)
            frontal_view = "AP" if frontal_path else ""

        lateral_path = choose(lat)
        report = clean_report(row["text"])

        records.append({
            "subject_id": str(row["subject_id"]),
            "study_id": study_id,
            "frontal_path": frontal_path,
            "frontal_view": frontal_view,
            "lateral_path": lateral_path,
            "text": report,
            "split": split_name
        })

    return pd.DataFrame(records)


train_processed = process_dataframe(train_df, "train")
val_processed = process_dataframe(val_df, "validation")

print("Processed train:", train_processed.shape)
print("Processed validation:", val_processed.shape)

In [ ]:
processed = pd.concat(
    [train_processed, val_processed],
    ignore_index=True
)

# فحص عدم وجود Study مكرر بين المجموعتين
train_studies = set(train_processed["study_id"])
val_studies = set(val_processed["study_id"])

overlap = train_studies & val_studies

print("Studies appearing in both splits:", len(overlap))

# حفظ ملفات المراجعة قبل استبعاد السجلات
train_processed.to_csv(
    os.path.join(OUTPUT_DIR, "train_studies_audit.csv"),
    index=False
)

val_processed.to_csv(
    os.path.join(OUTPUT_DIR, "validation_studies_audit.csv"),
    index=False
)

# استبعاد الفحوصات التي ينقصها أحد المدخلات المطلوبة
complete = processed[
    processed["frontal_path"].ne("")
    & processed["lateral_path"].ne("")
    & processed["text"].str.strip().ne("")
].copy()

# لو ظهر Study في المجموعتين، لا نحتفظ به تلقائيًا
if overlap:
    complete = complete[~complete["study_id"].isin(overlap)].copy()

# إزالة التكرارات المتطابقة على مستوى الـ split والـ Study
complete = complete.drop_duplicates(
    subset=["split", "study_id"]
).reset_index(drop=True)

final_columns = [
    "subject_id",
    "study_id",
    "frontal_path",
    "lateral_path",
    "text",
    "split"
]

complete = complete[final_columns]

final_path = os.path.join(
    OUTPUT_DIR,
    "chest_xray_frontal_lateral_reports.csv"
)

complete.to_csv(final_path, index=False)

print("\nFinal CSV:", final_path)
print("Final rows:", len(complete))
print("\nRows per split:")
print(complete["split"].value_counts())

print("\nPreview:")
display(complete.head())

print("\nMissing values:")
print(complete.isna().sum())

In [ ]:
print("Original CSV rows")
print("Train:", len(train_df))
print("Validation:", len(val_df))

print("\nProcessed rows")
print("Train:", len(train_processed))
print("Validation:", len(val_processed))

print("\nUnique subjects")
print("Train:", train_processed["subject_id"].nunique())
print("Validation:", val_processed["subject_id"].nunique())

print("\nImage availability in processed data")
for name, data in [
    ("Train", train_processed),
    ("Validation", val_processed)
]:
    print(f"\n{name}")
    print("Frontal exists:",
          data["frontal_path"].apply(os.path.isfile).sum())
    print("Lateral exists:",
          data["lateral_path"].apply(os.path.isfile).sum())
    print("Non-empty reports:",
          data["text"].str.strip().ne("").sum())

In [ ]:
import os

for name, data in [
    ("Train", train_processed),
    ("Validation", val_processed)
]:
    print(f"\n===== {name} =====")
    print("Total rows:", len(data))

    print(
        "Frontal paths exist:",
        data["frontal_path"].apply(
            lambda p: os.path.isfile(str(p))
        ).sum()
    )

    print(
        "Lateral paths exist:",
        data["lateral_path"].apply(
            lambda p: os.path.isfile(str(p))
        ).sum()
    )

    print(
        "Non-empty reports:",
        data["text"].fillna("").astype(str).str.strip().ne("").sum()
    )

    print("\nExample frontal path:")
    print(data["frontal_path"].iloc[0] if len(data) else "No rows")

    print("\nExample lateral path:")
    print(data["lateral_path"].iloc[0] if len(data) else "No rows")

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = False

CSV_PATH = (
    "/kaggle/working/processed_chest_xray/"
    "chest_xray_frontal_lateral_reports.csv"
)

IMG_SIZE = (224, 224)

df = pd.read_csv(CSV_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nSplit distribution:")
print(df["split"].value_counts())

df.head()

In [ ]:
def validate_image(path):
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


valid_rows = []
bad_frontal = 0
bad_lateral = 0

for row in tqdm(df.itertuples(index=False), total=len(df)):
    frontal_ok = (
        isinstance(row.frontal_path, str)
        and os.path.isfile(row.frontal_path)
        and validate_image(row.frontal_path)
    )

    lateral_ok = (
        isinstance(row.lateral_path, str)
        and os.path.isfile(row.lateral_path)
        and validate_image(row.lateral_path)
    )

    if not frontal_ok:
        bad_frontal += 1

    if not lateral_ok:
        bad_lateral += 1

    if frontal_ok and lateral_ok:
        valid_rows.append(row.Index if hasattr(row, "Index") else row)

print("Invalid frontal images:", bad_frontal)
print("Invalid lateral images:", bad_lateral)

In [ ]:
valid_indices = []
bad_frontal = 0
bad_lateral = 0

for idx, row in tqdm(df.iterrows(), total=len(df)):
    frontal_ok = (
        isinstance(row["frontal_path"], str)
        and os.path.isfile(row["frontal_path"])
        and validate_image(row["frontal_path"])
    )

    lateral_ok = (
        isinstance(row["lateral_path"], str)
        and os.path.isfile(row["lateral_path"])
        and validate_image(row["lateral_path"])
    )

    if not frontal_ok:
        bad_frontal += 1
    if not lateral_ok:
        bad_lateral += 1

    if frontal_ok and lateral_ok:
        valid_indices.append(idx)

df_valid = df.loc[valid_indices].reset_index(drop=True)

print("Original rows:", len(df))
print("Valid rows:", len(df_valid))
print("Invalid frontal:", bad_frontal)
print("Invalid lateral:", bad_lateral)
print(df_valid["split"].value_counts())

In [ ]:
import matplotlib.pyplot as plt

sample = df_valid.iloc[0]

def preprocess_image(path, size=IMG_SIZE):
    with Image.open(path) as img:
        img = img.convert("L")
        img = img.resize(size, Image.Resampling.BILINEAR)

        image_array = np.asarray(img, dtype=np.float32) / 255.0

    return image_array[..., np.newaxis]


frontal = preprocess_image(sample["frontal_path"])
lateral = preprocess_image(sample["lateral_path"])

print("Frontal shape:", frontal.shape)
print("Lateral shape:", lateral.shape)
print("Pixel range:", frontal.min(), frontal.max())

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(frontal.squeeze(), cmap="gray")
axes[0].set_title("Frontal X-ray")
axes[0].axis("off")

axes[1].imshow(lateral.squeeze(), cmap="gray")
axes[1].set_title("Lateral X-ray")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import re
import pandas as pd
import numpy as np

# Work on a copy of the valid dataset
text_df = df_valid.copy()

def clean_report_text(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Normalize line breaks and whitespace
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove repeated whitespace while preserving section structure
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n\s*\n+", "\n", text)
    text = text.strip()

    # Normalize common section headings
    text = re.sub(r"(?i)\bfindings\s*:", "Findings:", text)
    text = re.sub(r"(?i)\bimpression\s*:", "Impression:", text)

    return text

text_df["clean_text"] = text_df["text"].apply(clean_report_text)

# Remove empty reports
text_df = text_df[
    text_df["clean_text"].str.strip().ne("")
].copy()

text_df = text_df.reset_index(drop=True)

print("Rows after text cleaning:", len(text_df))
print("Empty reports:", text_df["clean_text"].eq("").sum())

print("\nOriginal report:\n")
print(text_df.loc[0, "text"][:1000])

print("\nCleaned report:\n")
print(text_df.loc[0, "clean_text"][:1000])

In [ ]:
START_TOKEN = "<start>"
END_TOKEN = "<end>"
UNK_TOKEN = "<unk>"

text_df["target_text"] = (
    START_TOKEN + " " +
    text_df["clean_text"] + " " +
    END_TOKEN
)

print(text_df["target_text"].iloc[0][:1000])

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

train_texts = text_df.loc[
    text_df["split"].eq("train"), "target_text"
].tolist()

val_texts = text_df.loc[
    text_df["split"].eq("validation"), "target_text"
].tolist()

print("Train reports:", len(train_texts))
print("Validation reports:", len(val_texts))

# Preserve <start>, <end>, and <unk> as special tokens
custom_filters = '!"#$%&()*+,-./:;=?@[\\]^`{|}~\t\n'

tokenizer = Tokenizer(
    filters=custom_filters,
    lower=False,
    oov_token=UNK_TOKEN
)

# Fit only on training reports
tokenizer.fit_on_texts(train_texts)

vocab_size = len(tokenizer.word_index) + 1

print("Vocabulary size:", vocab_size)
print("Start token ID:", tokenizer.word_index.get(START_TOKEN))
print("End token ID:", tokenizer.word_index.get(END_TOKEN))
print("Unknown token ID:", tokenizer.word_index.get(UNK_TOKEN))

In [ ]:
train_sequences = tokenizer.texts_to_sequences(train_texts)
val_sequences = tokenizer.texts_to_sequences(val_texts)

train_lengths = np.array(
    [len(seq) for seq in train_sequences]
)

MAX_LENGTH = min(
    300,
    max(10, int(np.percentile(train_lengths, 95)))
)

print("95th percentile length:", np.percentile(train_lengths, 95))
print("Selected MAX_LENGTH:", MAX_LENGTH)

print(
    "Train reports longer than limit:",
    np.mean(train_lengths > MAX_LENGTH) * 100,
    "%"
)

# Post-padding: pad at the end of each sequence
train_padded = pad_sequences(
    train_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

val_padded = pad_sequences(
    val_sequences,
    maxlen=MAX_LENGTH,
    padding="post",
    truncating="post"
)

print("Train padded shape:", train_padded.shape)
print("Validation padded shape:", val_padded.shape)

In [ ]:
sample_sequence = train_padded[0]

decoded_report = tokenizer.sequences_to_texts(
    [sample_sequence]
)[0]

print("Tokenized report:\n")
print(sample_sequence[:50])

print("\nDecoded report:\n")
print(decoded_report[:1000])

In [ ]:
import os
import json

OUTPUT_DIR = "/kaggle/working/processed_chest_xray"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save cleaned report dataset
text_df.to_csv(
    os.path.join(OUTPUT_DIR, "chest_xray_clean_reports.csv"),
    index=False
)

# Save tokenizer configuration
with open(
    os.path.join(OUTPUT_DIR, "tokenizer.json"),
    "w",
    encoding="utf-8"
) as f:
    f.write(tokenizer.to_json())

# Save padded sequences
np.save(
    os.path.join(OUTPUT_DIR, "train_text_sequences.npy"),
    train_padded
)

np.save(
    os.path.join(OUTPUT_DIR, "validation_text_sequences.npy"),
    val_padded
)

# Save metadata
metadata = {
    "vocab_size": vocab_size,
    "max_length": MAX_LENGTH,
    "start_token": START_TOKEN,
    "end_token": END_TOKEN,
    "unknown_token": UNK_TOKEN
}

with open(
    os.path.join(OUTPUT_DIR, "text_preprocessing_config.json"),
    "w",
    encoding="utf-8"
) as f:
    json.dump(metadata, f, indent=2)

print("Text preprocessing files saved to:", OUTPUT_DIR)
print("Vocabulary size:", vocab_size)
print("Maximum sequence length:", MAX_LENGTH)

In [ ]:
# %% [markdown]
# # 2 CNN (frontal + lateral) -> Transformer encoder -> Transformer decoder
# Flat code: no classes, no custom layers. (Needs Keras 3 = TF >= 2.16, default on Kaggle.)
# Needs from your notebook: text_df, tokenizer, train_padded, val_padded,
# vocab_size, MAX_LENGTH, START_TOKEN, END_TOKEN, UNK_TOKEN

# %%
import math
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

IMG_SIZE = 224
D_MODEL = 256
NUM_HEADS = 8
FF_DIM = 1024
NUM_ENC_LAYERS = 1
NUM_DEC_LAYERS = 3
DROPOUT = 0.1
BATCH_SIZE = 32
EPOCHS = 30
PEAK_LR = 3e-4

NUM_VIS = (IMG_SIZE // 16) ** 2      # 196 tokens per image (4 poolings: 224 -> 14)
FEAT = IMG_SIZE // 16                # 14

assert int(train_padded.max()) < vocab_size

# %% [markdown]
# ## Positional encodings (plain numpy constants)

# %%
i = np.arange(D_MODEL)[None, :]
denom = np.power(10000.0, (2 * (i // 2)) / D_MODEL)

pos = np.arange(NUM_VIS)[:, None]
pe_vis = np.where(i % 2 == 0, np.sin(pos / denom), np.cos(pos / denom)).astype("float32")

pos = np.arange(MAX_LENGTH - 1)[:, None]
pe_text = np.where(i % 2 == 0, np.sin(pos / denom), np.cos(pos / denom)).astype("float32")

# %% [markdown]
# ## CNN 1: frontal

# %%
frontal_in = keras.Input((IMG_SIZE, IMG_SIZE, 1), name="frontal")
x = frontal_in
for filters in (32, 64, 128, 256):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)                      # -> (14, 14, 256)

x = layers.Reshape((NUM_VIS, 256))(x)                 # 196 tokens
x = layers.Dense(D_MODEL)(x)
x = x + pe_vis
frontal_tokens = layers.LayerNormalization(epsilon=1e-6)(x)

# %% [markdown]
# ## CNN 2: lateral (same structure, its own weights)

# %%
lateral_in = keras.Input((IMG_SIZE, IMG_SIZE, 1), name="lateral")
x = lateral_in
for filters in (32, 64, 128, 256):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling2D()(x)

x = layers.Reshape((NUM_VIS, 256))(x)
x = layers.Dense(D_MODEL)(x)
x = x + pe_vis
lateral_tokens = layers.LayerNormalization(epsilon=1e-6)(x)

# %% [markdown]
# ## Transformer encoder (fuses both views)

# %%
memory = layers.Concatenate(axis=1)([frontal_tokens, lateral_tokens])   # (B, 392, 256)

for _ in range(NUM_ENC_LAYERS):
    h = layers.LayerNormalization(epsilon=1e-6)(memory)
    h = layers.MultiHeadAttention(NUM_HEADS, D_MODEL // NUM_HEADS, dropout=DROPOUT)(h, h)
    memory = layers.Add()([memory, layers.Dropout(DROPOUT)(h)])

    h = layers.LayerNormalization(epsilon=1e-6)(memory)
    h = layers.Dense(FF_DIM, activation="gelu")(h)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.Dense(D_MODEL)(h)
    memory = layers.Add()([memory, layers.Dropout(DROPOUT)(h)])

memory = layers.LayerNormalization(epsilon=1e-6)(memory)
encoder_model = keras.Model([frontal_in, lateral_in], memory, name="encoder")

# %% [markdown]
# ## Transformer decoder

# %%
tokens_in = keras.Input((MAX_LENGTH - 1,), dtype="int32", name="tokens")
memory_in = keras.Input((2 * NUM_VIS, D_MODEL), name="memory")

y = layers.Embedding(vocab_size, D_MODEL)(tokens_in)
y = y * math.sqrt(D_MODEL) + pe_text
y = layers.Dropout(DROPOUT)(y)

for _ in range(NUM_DEC_LAYERS):
    # masked self-attention (can't look at future words)
    h = layers.LayerNormalization(epsilon=1e-6)(y)
    h = layers.MultiHeadAttention(NUM_HEADS, D_MODEL // NUM_HEADS, dropout=DROPOUT)(
        h, h, use_causal_mask=True)
    y = layers.Add()([y, layers.Dropout(DROPOUT)(h)])

    # cross-attention (look at the X-ray tokens)
    h = layers.LayerNormalization(epsilon=1e-6)(y)
    h = layers.MultiHeadAttention(NUM_HEADS, D_MODEL // NUM_HEADS, dropout=DROPOUT)(
        h, memory_in)
    y = layers.Add()([y, layers.Dropout(DROPOUT)(h)])

    # feed-forward
    h = layers.LayerNormalization(epsilon=1e-6)(y)
    h = layers.Dense(FF_DIM, activation="gelu")(h)
    h = layers.Dropout(DROPOUT)(h)
    h = layers.Dense(D_MODEL)(h)
    y = layers.Add()([y, layers.Dropout(DROPOUT)(h)])

y = layers.LayerNormalization(epsilon=1e-6)(y)
logits = layers.Dense(vocab_size)(y)
decoder_model = keras.Model([tokens_in, memory_in], logits, name="decoder")

# %% [markdown]
# ## Full model

# %%
full_logits = decoder_model([tokens_in, encoder_model([frontal_in, lateral_in])])
model = keras.Model([frontal_in, lateral_in, tokens_in], full_logits, name="xray_report_transformer")
model.summary()

# %% [markdown]
# ## Data (the one small function tf.data needs for loading images)
# CHANGE the two column names to match your dataframe.

# %%
FRONTAL_COL = "frontal_path"
LATERAL_COL = "lateral_path"

train_df = text_df[text_df["split"] == "train"]
val_df = text_df[text_df["split"] == "validation"]
assert len(train_df) == len(train_padded) and len(val_df) == len(val_padded)


def load_pair(fp, lp, seq):
    imgs = []
    for p in (fp, lp):
        img = tf.io.decode_image(tf.io.read_file(p), channels=1, expand_animations=False)
        img.set_shape([None, None, 1])
        img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0
        imgs.append(img)
    return {"frontal": imgs[0], "lateral": imgs[1], "tokens": seq[:-1]}, seq[1:]


train_ds = (tf.data.Dataset.from_tensor_slices(
                (train_df[FRONTAL_COL].values, train_df[LATERAL_COL].values,
                 train_padded.astype("int32")))
            .shuffle(2048).map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
            .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

val_ds = (tf.data.Dataset.from_tensor_slices(
              (val_df[FRONTAL_COL].values, val_df[LATERAL_COL].values,
               val_padded.astype("int32")))
          .map(load_pair, num_parallel_calls=tf.data.AUTOTUNE)
          .batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE))

# %% [markdown]
# ## Compile + train
# `ignore_class=0` makes the loss skip padding.

# %%
steps_per_epoch = math.ceil(len(train_padded) / BATCH_SIZE)
lr = keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0,
    warmup_target=PEAK_LR,
    warmup_steps=steps_per_epoch,
    decay_steps=steps_per_epoch * (EPOCHS - 1),
)

model.compile(
    optimizer=keras.optimizers.AdamW(lr, weight_decay=1e-2, clipnorm=1.0),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True, ignore_class=0),
)

history = model.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=[
        keras.callbacks.ModelCheckpoint("/kaggle/working/best.weights.h5", monitor="val_loss",
                                        save_best_only=True, save_weights_only=True),
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=7, restore_best_weights=True),
    ],
)

# %% [markdown]
# ## Generate reports (greedy decoding)

# %%
batch_inputs, _ = next(iter(val_ds))
n = 4
fr = batch_inputs["frontal"][:n]
la = batch_inputs["lateral"][:n]

start_id = tokenizer.word_index[START_TOKEN]
end_id = tokenizer.word_index[END_TOKEN]
unk_id = tokenizer.word_index.get(UNK_TOKEN, 1)

memory_val = encoder_model([fr, la], training=False)       # images encoded once
tokens = np.zeros((n, MAX_LENGTH - 1), dtype="int32")
tokens[:, 0] = start_id
finished = np.zeros(n, dtype=bool)

for t in range(MAX_LENGTH - 2):
    step_logits = decoder_model([tf.constant(tokens), memory_val], training=False).numpy()[:, t, :]
    step_logits[:, 0] = -1e9            # never predict padding
    step_logits[:, unk_id] = -1e9       # never predict <unk>
    next_tok = step_logits.argmax(-1)
    next_tok[finished] = 0
    tokens[:, t + 1] = next_tok
    finished |= next_tok == end_id
    if finished.all():
        break

for k in range(n):
    words = []
    for tok in tokens[k]:
        if tok == end_id:
            break
        if tok not in (0, start_id):
            words.append(tokenizer.index_word.get(int(tok), ""))
    ref = [tokenizer.index_word.get(int(w), "") for w in val_padded[k]
           if w not in (0, start_id, end_id)]
    print(f"--- sample {k} ---\nREF : {' '.join(ref)}\nPRED: {' '.join(words)}\n")

In [ ]:
# %% [markdown]
# # Test the model (ground truth vs prediction) + save it
# Run AFTER the training cell of xray_transformer_simple.py (same notebook session).

# %% [markdown]
# ## 1) Generate reports for the whole validation set

# %%
import matplotlib.pyplot as plt

start_id = tokenizer.word_index[START_TOKEN]
end_id = tokenizer.word_index[END_TOKEN]
unk_id = tokenizer.word_index.get(UNK_TOKEN, 1)

all_frontal, all_lateral = [], []
pred_texts, true_texts = [], []
row = 0                                              # index into val_padded (val_ds is not shuffled)

for batch_inputs, _ in val_ds:
    fr = batch_inputs["frontal"]
    la = batch_inputs["lateral"]
    n = fr.shape[0]

    memory_val = encoder_model([fr, la], training=False)
    tokens = np.zeros((n, MAX_LENGTH - 1), dtype="int32")
    tokens[:, 0] = start_id
    finished = np.zeros(n, dtype=bool)

    for t in range(MAX_LENGTH - 2):
        step_logits = decoder_model([tf.constant(tokens), memory_val], training=False).numpy()[:, t, :]
        step_logits[:, 0] = -1e9                     # never predict padding
        step_logits[:, unk_id] = -1e9                # never predict <unk>
        next_tok = step_logits.argmax(-1)
        next_tok[finished] = 0
        tokens[:, t + 1] = next_tok
        finished |= next_tok == end_id
        if finished.all():
            break

    for k in range(n):
        pred_words = []
        for tok in tokens[k]:
            if tok == end_id:
                break
            if tok not in (0, start_id):
                pred_words.append(tokenizer.index_word.get(int(tok), ""))
        true_words = []
        for tok in val_padded[row + k]:
            if tok == end_id:
                break
            if tok not in (0, start_id):
                true_words.append(tokenizer.index_word.get(int(tok), ""))
        pred_texts.append(" ".join(pred_words))
        true_texts.append(" ".join(true_words))

    all_frontal.append(fr.numpy()[:8])               # keep a few images just for display
    all_lateral.append(la.numpy()[:8])
    row += n

print("Generated", len(pred_texts), "reports")

# %% [markdown]
# ## 2) See images + ground truth vs model output

# %%
NUM_SHOW = 5
imgs_f = np.concatenate(all_frontal)
imgs_l = np.concatenate(all_lateral)

for k in range(min(NUM_SHOW, len(imgs_f))):
    fig, ax = plt.subplots(1, 2, figsize=(7, 3.5))
    ax[0].imshow(imgs_f[k][:, :, 0], cmap="gray"); ax[0].set_title("Frontal"); ax[0].axis("off")
    ax[1].imshow(imgs_l[k][:, :, 0], cmap="gray"); ax[1].set_title("Lateral"); ax[1].axis("off")
    plt.show()
    print(f"[{k}] GROUND TRUTH:\n{true_texts[k]}\n")
    print(f"[{k}] MODEL OUTPUT:\n{pred_texts[k]}\n")
    print("=" * 100)

# %% [markdown]
# ## 3) BLEU score (optional)

# %%
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

refs = [[t.split()] for t in true_texts]
hyps = [p.split() for p in pred_texts]
smooth = SmoothingFunction().method1

print("BLEU-1:", round(corpus_bleu(refs, hyps, weights=(1, 0, 0, 0), smoothing_function=smooth), 4))
print("BLEU-2:", round(corpus_bleu(refs, hyps, weights=(.5, .5, 0, 0), smoothing_function=smooth), 4))
print("BLEU-3:", round(corpus_bleu(refs, hyps, weights=(1/3, 1/3, 1/3, 0), smoothing_function=smooth), 4))
print("BLEU-4:", round(corpus_bleu(refs, hyps, weights=(.25, .25, .25, .25), smoothing_function=smooth), 4))

# %% [markdown]
# ## 4) Save the model

# %%
SAVE_DIR = "/kaggle/working/processed_chest_xray"

model.save(SAVE_DIR + "/xray_report_model.keras")            # full model (architecture + weights)
encoder_model.save(SAVE_DIR + "/encoder.keras")               # for fast inference
decoder_model.save(SAVE_DIR + "/decoder.keras")
model.save_weights(SAVE_DIR + "/xray_report.weights.h5")      # weights only

print("Saved to:", SAVE_DIR)

# %% [markdown]
# ## 5) Load it back and check it matches (use this in a new notebook)
# The tokenizer.json + text_preprocessing_config.json you saved earlier are needed too.

# %%
loaded_encoder = keras.models.load_model(SAVE_DIR + "/encoder.keras", compile=False)
loaded_decoder = keras.models.load_model(SAVE_DIR + "/decoder.keras", compile=False)

test_mem_a = encoder_model([fr[:2], la[:2]], training=False).numpy()
test_mem_b = loaded_encoder([fr[:2], la[:2]], training=False).numpy()
print("Encoder outputs identical after loading:", np.allclose(test_mem_a, test_mem_b, atol=1e-5))

# NOTE: to rebuild the tokenizer later:
#   from tensorflow.keras.preprocessing.text import tokenizer_from_json
#   tokenizer = tokenizer_from_json(open(SAVE_DIR + "/tokenizer.json").read())

In [ ]:

import numpy as np
import tensorflow as tf
from rouge_score import rouge_scorer
import pandas as pd

# ==========================================
# 1) Generate predictions for a dataset
# ==========================================

def generate_reports(dataset, dataset_padded, dataset_name="Dataset"):
    start_id = tokenizer.word_index[START_TOKEN]
    end_id = tokenizer.word_index[END_TOKEN]
    unk_id = tokenizer.word_index.get(UNK_TOKEN, 1)

    pred_texts = []
    true_texts = []
    row = 0

    for batch_inputs, _ in dataset:
        fr = batch_inputs["frontal"]
        la = batch_inputs["lateral"]
        n = int(fr.shape[0])

        memory = encoder_model([fr, la], training=False)

        tokens = np.zeros(
            (n, MAX_LENGTH - 1),
            dtype="int32"
        )
        tokens[:, 0] = start_id
        finished = np.zeros(n, dtype=bool)

        for t in range(MAX_LENGTH - 2):
            logits = decoder_model(
                [tf.constant(tokens), memory],
                training=False
            ).numpy()[:, t, :]

            logits[:, 0] = -1e9
            logits[:, unk_id] = -1e9

            next_tok = logits.argmax(axis=-1)
            next_tok[finished] = 0
            tokens[:, t + 1] = next_tok

            finished |= (next_tok == end_id)

            if finished.all():
                break

        # Decode predictions and ground truth
        for k in range(n):
            pred_words = []

            for tok in tokens[k]:
                if tok == end_id:
                    break
                if tok not in (0, start_id):
                    pred_words.append(
                        tokenizer.index_word.get(int(tok), "")
                    )

            true_words = []

            for tok in dataset_padded[row + k]:
                if tok == end_id:
                    break
                if tok not in (0, start_id):
                    true_words.append(
                        tokenizer.index_word.get(int(tok), "")
                    )

            pred_texts.append(" ".join(pred_words))
            true_texts.append(" ".join(true_words))

        row += n

    print(f"{dataset_name}: generated {len(pred_texts)} reports")
    return true_texts, pred_texts


# ==========================================
# 2) Calculate ROUGE metrics
# ==========================================

def calculate_rouge(true_texts, pred_texts):
    scorer = rouge_scorer.RougeScorer(
        ["rouge1", "rouge2", "rougeL"],
        use_stemmer=False
    )

    metric_names = ["rouge1", "rouge2", "rougeL"]
    results = {}

    for metric in metric_names:
        scores = [
            scorer.score(ref, pred)[metric]
            for ref, pred in zip(true_texts, pred_texts)
        ]

        results[metric] = {
            "Precision": np.mean([s.precision for s in scores]),
            "Recall": np.mean([s.recall for s in scores]),
            "F1": np.mean([s.fmeasure for s in scores]),
        }

    return results


# ==========================================
# 3) Evaluate training and validation
# ==========================================

# IMPORTANT:
# Replace train_padded with the padded ground-truth
# sequences corresponding to train_ds, in the same order.
# val_padded is already used in your validation code.

train_true, train_pred = generate_reports(
    train_ds, train_padded, "Training"
)

val_true, val_pred = generate_reports(
    val_ds, val_padded, "Validation"
)

train_rouge = calculate_rouge(train_true, train_pred)
val_rouge = calculate_rouge(val_true, val_pred)


# ==========================================
# 4) Display comparison
# ==========================================

rows = []

for metric in ["rouge1", "rouge2", "rougeL"]:
    for score_type in ["Precision", "Recall", "F1"]:
        tr = train_rouge[metric][score_type]
        va = val_rouge[metric][score_type]

        rows.append({
            "Metric": metric.upper(),
            "Score": score_type,
            "Train": tr,
            "Validation": va,
            "Difference (Train - Val)": tr - va
        })

results_df = pd.DataFrame(rows)

print("\nROUGE: Training vs Validation")
display(
    results_df.style.format({
        "Train": "{:.4f}",
        "Validation": "{:.4f}",
        "Difference (Train - Val)": "{:+.4f}"
    })
)
